## Section 1: Regression (Taxi)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
RANDOM_STATE = 42
TAXI_URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/pu9kbeSaAtRZ7RxdJKX9_A/yellow-tripdata.csv'

# 1. Loading and Inspection
taxi = pd.read_csv(TAXI_URL)
print(f"Shape: {taxi.shape}")
display(taxi.head())
taxi.info()
display(taxi.describe())
print("\nMissing values per column:")
print(taxi.isna().sum())

# Visualises Tip Amount Distribution
plt.figure(figsize=(10, 5))
sns.histplot(taxi['tip_amount'], bins=50, kde=True)
plt.title('Distribution of tip_amount')
plt.xlabel('Tip Amount ($)')
plt.show()

# Filters out non-positive distances/fares and negative tips
taxi = taxi.dropna()
taxi = taxi[(taxi['trip_distance'] > 0) & (taxi['fare_amount'] > 0)]
taxi = taxi[taxi['tip_amount'] >= 0]

# 4. Feature engineering
# Creating 'total_surcharges' (improvement_surcharge + mta_tax + tolls_amount)
taxi['total_surcharges'] = taxi['mta_tax'] + taxi['tolls_amount'] + taxi['improvement_surcharge']

# 5. Encoding and Feature Selection
# Encoding categorical columns using pd.get_dummies
taxi_encoded = pd.get_dummies(taxi, columns=['VendorID', 'RatecodeID', 'payment_type', 'store_and_fwd_flag'], drop_first=True)

# 6. Separate features and target
X = taxi_encoded.drop(columns=['tip_amount'])
y = taxi_encoded['tip_amount']

# 7. Splits data into train / validation / test (60/20/20)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=RANDOM_STATE)

# 8. Scale numeric features (fit on TRAIN only)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# 9. Trains Model
model = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=RANDOM_STATE)
model.fit(X_train_scaled, y_train)

# 10. Predicts and Reports Performance
def evaluate_reg(X_scaled, y_true, split_name):
    y_pred = model.predict(X_scaled)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"{split_name} Performance")
    print(f"RMSE: ${rmse:.2f}")
    print(f"R²:   {r2:.3f}\n")
    return y_pred

y_train_pred = evaluate_reg(X_train_scaled, y_train, "Training")
y_val_pred = evaluate_reg(X_val_scaled, y_val, "Validation")
y_test_pred = evaluate_reg(X_test_scaled, y_test, "Test")

# 11. Plots predicted vs actual for Test Set
plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_test_pred, alpha=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Tip Amount ($)')
plt.ylabel('Predicted Tip Amount ($)')
plt.title('Actual vs Predicted Tip Amount (Test Set)')
plt.show()

## Student Reflection for Section 1

###Taxi Data Exploration

The dataset contains 41,202 rows and 13 columns. With respect to missing / impossible values, there were no missing values in the raw dataset. There were impossible values present in the data, such as non-positive trip distances, zero and negative fare amounts.

The 'tip_amount' distribution is right-skewed, and has majority of tips being small, often ranging from $0 to $5. There are extreme outliers which also exist in this column, values extending up to $50 and above.

The RandomForestRegressor was used due to the skew and outliers, since it is relatively robust to outliers. The data was filtered to ensure that the model only learns from valid, and realistic trip records.


###Taxi Data Preprocessing

Rows where the 'trip_distance', or the 'fare_amount' were less than 0 and the tip_amount was negative, were filtered out.

The 'total_surchages' column was created by adding the columns 'mta_tax', 'tolls_amount', and 'improvement_surcharge'columns.

StandardScaler was used for scaling. This is because, linear models perform more consistently when numeric features are on the same scale.

A 60/20/20 splitting strategy was used, 80% is kept for training the data, with 20% of the 80% being used for validation. 20% of the data is kept for testing.
The validation set is used for fine-tuning hyper-parameters. The test data is kept until the end of the training, to check that the model is able to accurately predict the labels for new data and provide an unbiased estimate for real-world scenarios.

The RandomForestRegressor with a max_depth of 10, was used for evaluation.

The results of the performance metrics are stated below:

Training Performance
RMSE: $4.60
(R²:   0.130)

Validation Performance
RMSE: $5.07
(R²:   0.035)

Test Performance
RMSE: $5.04
(R²:   0.041)

This shows that the model is slightly overfitting, as there is a gap between the performance metric from the training data and that of the test data, as the R² drops from 0.13 to 0.041.

To reduce this overfitting, I will reduce the max_depth value to 7 or 8 or increase the min_samples_leaf to force the model generalize better.





## Section 2: Supervised Learning (Obesity dataset)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

# 1. Load the obesity dataset
OBESITY_URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/GkDzb7bWrtvGXdPOfk6CIg/Obesity-level-prediction-dataset.csv'
obesity = pd.read_csv(OBESITY_URL)

# 2. Inspect shape, head, info, describe, and missing values
print(f"Shape: {obesity.shape}")
display(obesity.head())
obesity.info()
display(obesity.describe())
print("\nMissing Values:")
print(obesity.isnull().sum())

# 3. Show class distribution
print("\nClass Distribution:")
print(obesity['NObeyesdad'].value_counts(normalize=True)*100)

plt.figure(figsize=(12, 5))
sns.countplot(data=obesity, x='NObeyesdad', palette='viridis')
plt.title('Distribution of Obesity Levels')
plt.xticks(rotation=45)
plt.show()

# 4. Feature Engineering: BMI
obesity['BMI'] = obesity['Weight'] / (obesity['Height']**2)

# 5. Encode Categorical Columns
binary_cols = ['family_history_with_overweight', 'FAVC', 'SMOKE', 'SCC']
for col in binary_cols:
    obesity[col] = obesity[col].map({'yes': 1, 'no': 0})

encoded_df = pd.get_dummies(obesity, columns=['Gender', 'MTRANS', 'CAEC', 'CALC'], drop_first=True)

# 6. Encode the target into integer labels
le = LabelEncoder()
encoded_df['NObeyesdad'] = le.fit_transform(encoded_df['NObeyesdad'])

# 7. Split data
X = encoded_df.drop(columns=['NObeyesdad'])
y = encoded_df['NObeyesdad']

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=42)

# 8. Scale numeric features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# 9. Import, initialise, and train a classifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

# 10. Predict and Report Accuracy and Macro-F1 for ALL THREE sets
def evaluate_split(X_scaled, y_true, split_name):
    y_pred = model.predict(X_scaled)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro')
    print(f"{split_name} Performance")
    print(f"Accuracy: {acc:.3f}")
    print(f"Macro-F1: {f1:.3f}\n")
    return y_pred

y_train_pred = evaluate_split(X_train_scaled, y_train, "TRAIN")
y_val_pred = evaluate_split(X_val_scaled, y_val, "VALIDATION")
y_test_pred = evaluate_split(X_test_scaled, y_test, "TEST")

# Detailed Classification Report for the Test set
print("Detailed Test Classification Report:")
print(classification_report(y_test, y_test_pred, target_names=le.classes_))

# 11. Show a confusion matrix for the test set
cm_test = confusion_matrix(y_test, y_test_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_test, display_labels=le.classes_)
fig, ax = plt.subplots(figsize=(10,8))
disp.plot(ax=ax, cmap='Blues', xticks_rotation='vertical')
plt.title("Test Set Confusion Matrix")
plt.show()

# Reasoning Box for Section 2

## Analysis

The dataset contains 2,111 rows and 17 features. There are 8 numeric columns and 9 categorical columns.

With regards to class balance, the target column, 'NObeyesdad' is balanced across 7 categories with each category representing approximately 13-16% of the data. The class balance is important because, high imbalance can cause the model to ignore other minor classes to achieve high accuracy.

A 60/20/20 splitting ratio was used.
Using 'stratify=y' is significant because it ensures that each split (train, validation, test) maintains the same distribution of obesity levels across the dataset. Without stratifying, a split might accidentally result in a training set missing a class or having a different ratio than the test set, leading to poor evaluation and poor performance metrics.

The RandomForestRegressor was used because of the skew and outliers. This is because it is  robust to outliers.

## Results

Training Performance
Accuracy:  1.000
Macro-F1:  1.000

Validation Performance
Accuracy:  0.931
Macro-F1:  0.929

Test Performance
Accuracy:  0.939
Macro-F1:  0.939

The results show that the model had a 100% accuracy score, when predicting labels for data during training. For new data, the model was able to predict the appropriate labels with an accuracy score of about 94%, which is also very good. Meaning for every new patient the model can accurately predict  the appropriate labels, with an accuracy of about 94%.

Technically, the model is slightly overfitting, this is because, there's a 6-7% gap between the scores for the Training Performance and the Validation and Test Performance.

Based on the Confusion Matrix, the hardest obesity levels to tell apart are "Obesity Level I" and "Obesity Level II". This may be because the BMI boundaries are very close, makng it very difficult for the model to tell them apart, and identify individuals that sit on the boundary.

## Unsupervised Learning (Obesity Dataset)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Loads dataset for unsupervised learning task
df = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/GkDzb7bWrtvGXdPOfk6CIg/Obesity-level-prediction-dataset.csv')

# Removes missing values
df.dropna()

# Stores true labels for later use
true_labels = df['NObeyesdad']

# Drop the target column with the true labels and store out X and Y labels, to ensure unsupervised learning integrity
X = df.drop(columns=['NObeyesdad'])

#Encode and scale the data
encodedX = pd.get_dummies(X, drop_first=True)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(encodedX)

# Finding K

inertia = []
silhouette_scores = []

k_range = range(2, 10)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, kmeans.labels_))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Elbow Plot
ax1.plot(k_range, inertia, marker='o', linestyle='--')
ax1.set_title('The Elbow Method')
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('Inertia (WCSS)')

# Silhouette Plot
ax2.plot(k_range, silhouette_scores, marker='o', linestyle='--')
ax2.set_title('Silhouette Coefficients')
ax2.set_xlabel('Number of Clusters (k)')
ax2.set_ylabel('Silhouette Score')
plt.show()

#Fitting K-Means with Chosen k value

chosenk_value = 3
final_kmeans = KMeans(n_clusters=chosenk_value, random_state=42, n_init=10)
cluster_assignments = final_kmeans.fit_transform(X_scaled)
df['Cluster'] = final_kmeans.labels_

# PCA Visualization

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=df['Cluster'], palette='Set1', alpha=0.7)
plt.title(f'K-Means Clusters Visualized via 2D PCA (k={chosenk_value})')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend(title='Cluster')
plt.show()


# Contingency Matrix

print("Contingency Matrix")
crosstab = pd.crosstab(true_labels, df['Cluster'])
print(crosstab)

# Section 3 Reflection: Unsupervised Learning (Clustering)


I chose the value 3 for k, because the Elbow plot showed a sharp decrease in inertia until k = 3, after which the rate of increase slowed down. The Silhouette score peaked at k = 2 and stayed relatively at k = 3 before dropping significantly for higher values of k, suggesting three different underlying patterns in the feature space.

The clusters partially resemble real obesity levels but do not map perfectly to the 7 specific categories. Cluster 1 strongly captures 'Obesity Type III' and 'Obesity Type II' representing a 'High BMI' group. Cluster 0 seems to group 'Insufficient' and 'Normal' weights. The breakdown happens in the 'Overweight' and 'Obesity Type I' levels which are split across Clusters 1 and 2.

In a public-health setting where labels are expensive, these clusters could be used for 'Targeted Intervention'. Health officials could identify the 'High-Risk' cluster (Cluster 1) to prioritize resources or clinical screenings for those individuals, even without knowing their exact clinical diagnosis. It also helps in 'Stratified Sampling', ensuring that expensive label collection covers representatives from all naturally occurring sub-groups in the population.

## Section 4

The classifier was able to learn the difference between categories with very close boundaries. However the K-Means was not able to distinguish the difference between labels that are very close.

Evaluating the continuous variable involved measuring the magnitude of error using RMSE, and the proportion of variance using R².

The biggest train-vs-test gap occurred in Section 2, where the model achieved a perfect 1.0 accuracy on training data but dropped to 0.939 on the test set. The single most effective way to close this gap would be to implement model regularization, specifically by limiting the max_depth of the Random Forest or increasing min_samples_leaf to prevent individual trees from memorizing specific training instances/noise.